In [0]:
# Magic command to install dependencies directly on the driver
%pip install faker

import os
import random
import pandas as pd
from datetime import datetime, timedelta
from faker import Faker

# ==========================================
# 1. CONFIGURATION (From your config.py)
# ==========================================
CUSTOMER_COUNT = 1000
POLICY_COUNT = 1000
AGENT_COUNT = 300
MONEY_IN_COUNT = 5000

SOURCE_SYSTEM = "DB2_INSURANCE"
BATCH_ID = "BATCH_001"

# Databricks storage path mapped for flat file writes
RAW_DATA_PATH = "/dbfs/FileStore/tables/raw_data"
os.makedirs(RAW_DATA_PATH, exist_ok=True)

# File Paths mapped precisely to output as single parquet files
PRODUCT_PATH = f"{RAW_DATA_PATH}/product_master.parquet"
COMMISSION_PATH = f"{RAW_DATA_PATH}/product_commission_rule.parquet"
CUSTOMER_PATH = f"{RAW_DATA_PATH}/customer.parquet"
CUSTOMER_ROLE_PATH = f"{RAW_DATA_PATH}/customer_role_master.parquet"
AGENT_PATH = f"{RAW_DATA_PATH}/agent.parquet"
POLICY_PATH = f"{RAW_DATA_PATH}/policy.parquet"
CUSTOMER_POLICY_PATH = f"{RAW_DATA_PATH}/customer_policy.parquet"
AGENT_POLICY_PATH = f"{RAW_DATA_PATH}/agent_policy.parquet"
MONEY_IN_PATH = f"{RAW_DATA_PATH}/money_in_dtl.parquet"

CITIES = [
    "Delhi", "Mumbai", "Kolkata", "Bhopal", "Indore", 
    "Pune", "Bangalore", "Hyderabad", "Gondia", "Bhandara", 
    "Nagpur", "Balaghat", "Seoni", "Jabalpur", "Wardha"
]

fake = Faker('en_IN') # Using India locale for context-accurate data
current_time = datetime.now()

# ==========================================
# 2. SEED MASTERS (Products)
# ==========================================
products = [
    {"product_id": "P001", "product_name": "Term Life Plan", "category": "LIFE"},
    {"product_id": "P002", "product_name": "Endowment Wealth", "category": "LIFE"},
    {"product_id": "P003", "product_name": "Health Shield Pro", "category": "HEALTH"},
    {"product_id": "P004", "product_name": "Motor Comprehensive", "category": "GENERAL"}
]

commission_rules = [
    {"rule_id": "R001", "product_id": "P001", "commission_percentage": 15.0},
    {"rule_id": "R002", "product_id": "P002", "commission_percentage": 10.0},
    {"rule_id": "R003", "product_id": "P003", "commission_percentage": 12.5},
    {"rule_id": "R004", "product_id": "P004", "commission_percentage": 7.5}
]

# ==========================================
# 3. GENERATION ENGINE FUNCTIONS
# ==========================================

def generate_customers():
    data = []
    for i in range(1, CUSTOMER_COUNT + 1):
        data.append({
            "customer_no": i,
            "source_customer_no": f"SRC_CUST_{20000+i}",
            "customer_type": random.choice(["INDIVIDUAL", "CORPORATE"]),
            "customer_status": random.choices(["ACTIVE", "INACTIVE"], weights=[0.90, 0.10])[0],
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.email(),
            "phone": fake.msisdn()[:10],
            "source_system": SOURCE_SYSTEM,
            "batch_id": BATCH_ID,
            "created_date": current_time
        })
    df = pd.DataFrame(data)
    df.to_parquet(CUSTOMER_PATH, index=False)
    return df

def generate_customer_roles():
    data = []
    for i in range(1, CUSTOMER_COUNT + 1):
        data.append({
            "customer_no": i,
            "role_type": random.choices(["PROPOSER", "INSURED", "BENEFICIARY"], weights=[0.70, 0.20, 0.10])[0],
            "source_system": SOURCE_SYSTEM,
            "batch_id": BATCH_ID,
            "created_date": current_time
        })
    df = pd.DataFrame(data)
    df.to_parquet(CUSTOMER_ROLE_PATH, index=False)
    return df

def generate_agents():
    data = []
    for i in range(1, AGENT_COUNT + 1):
        joining_date = current_time - timedelta(days=random.randint(30, 2000))
        agent_status = random.choices(["ACTIVE", "INACTIVE"], weights=[0.85, 0.15])[0]
        city = random.choice(CITIES)
        
        data.append({
            "agent_no": i,
            "source_agent_no": f"SRC_AGT_{10000+i}",
            "first_name": fake.first_name(),
            "middle_name": fake.first_name(),
            "last_name": fake.last_name(),
            "dob": fake.date_of_birth(minimum_age=25, maximum_age=60),
            "gender": random.choice(["MALE", "FEMALE"]),
            "mobile_no": fake.msisdn()[:10],
            "email": fake.email(),
            "joining_date": joining_date,
            "termination_date": None if agent_status == "ACTIVE" else joining_date + timedelta(days=random.randint(30, 500)),
            "designation": random.choice(["Insurance Advisor", "Senior Advisor", "Sales Manager"]),
            "branch_code": f"BR{random.randint(100, 999)}",
            "branch_name": f"{city} Branch",
            "city": city,
            "state": fake.state(),
            "country": "India",
            "agent_status": agent_status,
            "source_system": SOURCE_SYSTEM,
            "batch_id": BATCH_ID,
            "created_date": current_time,
            "created_by": "DATA_GENERATOR"
        })
    df = pd.DataFrame(data)
    df.to_parquet(AGENT_PATH, index=False)
    return df

def generate_policies():
    data = []
    for i in range(1, POLICY_COUNT + 1):
        data.append({
            "policy_no": i,
            "source_policy_no": f"SRC_POL_{50000+i}",
            "product_id": random.choice(products)["product_id"],
            "policy_status": random.choices(["INFORCE", "LAPSED", "TERMINATED"], weights=[0.80, 0.15, 0.05])[0],
            "sum_assured": random.choice([500000, 1000000, 2500000, 5000000]),
            "source_system": SOURCE_SYSTEM,
            "batch_id": BATCH_ID,
            "created_date": current_time
        })
    df = pd.DataFrame(data)
    df.to_parquet(POLICY_PATH, index=False)
    return df

def generate_cross_references(cust_df, agent_df, pol_df):
    cust_ids = cust_df["customer_no"].tolist()
    agent_ids = agent_df["agent_no"].tolist()
    policy_ids = pol_df["policy_no"].tolist()

    cust_policy = []
    agent_policy = []

    # Bind every policy to a random customer and agent to preserve referential integrity
    for p_id in policy_ids:
        cust_policy.append({
            "policy_no": p_id,
            "customer_no": random.choice(cust_ids),
            "relationship_type": "OWNER",
            "source_system": SOURCE_SYSTEM,
            "batch_id": BATCH_ID
        })
        agent_policy.append({
            "policy_no": p_id,
            "agent_no": random.choice(agent_ids),
            "split_percentage": 100.0,
            "source_system": SOURCE_SYSTEM,
            "batch_id": BATCH_ID
        })

    cp_df = pd.DataFrame(cust_policy)
    ap_df = pd.DataFrame(agent_policy)
    
    cp_df.to_parquet(CUSTOMER_POLICY_PATH, index=False)
    ap_df.to_parquet(AGENT_POLICY_PATH, index=False)
    return cp_df, ap_df

def generate_money_in(pol_df):
    policy_ids = pol_df["policy_no"].tolist()
    data = []
    
    for i in range(1, MONEY_IN_COUNT + 1):
        data.append({
            "payment_id": i,
            "policy_no": random.choice(policy_ids),
            "payment_type": random.choice(["RENEWAL", "NEW_BUSINESS", "TOP_UP"]),
            "premium_amount": round(random.uniform(5000.0, 75000.0), 2),
            "payment_status": random.choices(["SUCCESS", "FAILED", "PENDING"], weights=[0.92, 0.06, 0.02])[0],
            "payment_date": current_time - timedelta(days=random.randint(1, 365)),
            "source_system": SOURCE_SYSTEM,
            "batch_id": BATCH_ID,
            "created_date": current_time
        })
    df = pd.DataFrame(data)
    df.to_parquet(MONEY_IN_PATH, index=False)
    return df

# ==========================================
# 4. EXECUTION PIPELINE
# ==========================================

print("🚀 Starting relational data generation process...")

# Write Master files out first
pd.DataFrame(products).to_parquet(PRODUCT_PATH, index=False)
pd.DataFrame(commission_rules).to_parquet(COMMISSION_PATH, index=False)
print(" - Product & Commission Masters Written.")

# Run Independent Entity Generation
df_cust = generate_customers()
df_role = generate_customer_roles()
print(f" - Generated {len(df_cust)} Customers & Customer Roles.")

df_agent = generate_agents()
print(f" - Generated {len(df_agent)} Agents.")

df_policy = generate_policies()
print(f" - Generated {len(df_policy)} Base Policies.")

# Run Dependent Relational Table Generation
df_cp, df_ap = generate_cross_references(df_cust, df_agent, df_policy)
print(" - Generated Customer-Policy & Agent-Policy intersection tables.")

df_money = generate_money_in(df_policy)
print(f" - Generated {len(df_money)} Premium Transaction Ledger Records.")

print("\n💥 Success! All datasets generated cleanly as single flat parquet files inside:")
print(f"📂 Location: {RAW_DATA_PATH}\n")

# Quick terminal confirmation UI check on final output ledger
display(spark.createDataFrame(df_money.head(10)))